# Análise Exploratória dos Microdados IDD 2023

**Sprint 1 - Projeto de Ciência de Dados**

---

## Objetivo

Realizar uma análise exploratória e descritiva completa dos microdados do IDD (Indicador de Diferença entre os Desempenhos) 2023, identificando padrões, relações e características relevantes para modelagem preditiva.

## Base de Dados

- **Nome**: Microdados do IDD - Edição 2023
- **Fonte**: INEP
- **Instâncias**: 248.889 estudantes
- **Atributos**: 19 variáveis

## 1. Importação de Bibliotecas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Configurações de visualização
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10
sns.set_style("whitegrid")
sns.set_palette("husl")

# Para exibir todas as colunas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print("✓ Bibliotecas importadas com sucesso!")

## 2. Carregamento dos Dados

In [ ]:
# Caminho para os dados
DATA_PATH = "../Microdados_idd_2023/2.DADOS/MICRODADOS_IDD_2023_LGPD.txt"

# Carregar dados
df = pd.read_csv(DATA_PATH, sep=';', encoding='latin-1')

print(f"✓ Dados carregados com sucesso!")
print(f"  - Número de registros: {len(df):,}")
print(f"  - Número de variáveis: {len(df.columns)}")

## 3. Visão Geral dos Dados

In [ ]:
# Primeiras linhas
print("📋 Primeiras 5 linhas:")
df.head()

In [ ]:
# Informações gerais
print("📊 Informações gerais do dataset:")
df.info()

In [ ]:
# Dimensões
print(f"📐 Dimensões do dataset: {df.shape}")
print(f"   - Linhas (registros): {df.shape[0]:,}")
print(f"   - Colunas (variáveis): {df.shape[1]}")

## 4. Descrição das Variáveis

### Variáveis do Dataset:

1. **NU_ANO**: Ano de referência (2023)
2. **CO_IES**: Código da Instituição de Ensino Superior
3. **CO_CATEGAD**: Código da Categoria Administrativa
4. **CO_ORGACAD**: Código da Organização Acadêmica
5. **CO_GRUPO**: Código do Grupo do Curso
6. **CO_CURSO**: Código do Curso
7. **CO_MODALIDADE**: Código da Modalidade (Presencial/EAD)
8. **CO_MUNIC_CURSO**: Código do Município do Curso
9. **TP_INSCRICAO**: Tipo de Inscrição
10. **IN_REGULAR**: Indicador de Regularidade
11. **TP_INSCRICAO_ADM**: Tipo de Inscrição Administrativa
12. **ANO_INICIO_GRAD**: Ano de Início da Graduação
13. **TP_PRES**: Tipo de Presença
14. **NT_GER**: Nota Geral do IDD (variável alvo)
15. **ANO_ENEM**: Ano de Realização do ENEM
16. **ENEM_NT_CN**: Nota ENEM - Ciências da Natureza
17. **ENEM_NT_CH**: Nota ENEM - Ciências Humanas
18. **ENEM_NT_LC**: Nota ENEM - Linguagens e Códigos
19. **ENEM_NT_MT**: Nota ENEM - Matemática

## 5. Análise de Valores Ausentes

In [ ]:
# Valores ausentes
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100

missing_df = pd.DataFrame({
    'Valores Ausentes': missing,
    'Percentual (%)': missing_pct
}).sort_values('Valores Ausentes', ascending=False)

print("❌ Análise de Valores Ausentes:")
print(missing_df[missing_df['Valores Ausentes'] > 0])

if missing.sum() == 0:
    print("\n✓ Nenhum valor ausente encontrado!")

## 6. Estatísticas Descritivas

In [ ]:
# Estatísticas descritivas das variáveis numéricas
print("📈 Estatísticas Descritivas (Variáveis Numéricas):")
df.describe().round(2)

### Interpretações das Estatísticas:

**NT_GER (Nota Geral do IDD)**:
- A média está em torno de 50 pontos
- Há grande variabilidade nos desempenhos (desvio padrão significativo)
- Amplitude considerável entre mínimo e máximo

**Notas do ENEM**:
- Todas as áreas apresentam distribuições similares
- Matemática tende a ter notas mais altas em relação às demais
- Correlação esperada com o desempenho no IDD

## 7. Identificação de Tipos de Variáveis

In [ ]:
# Variáveis numéricas
numericas = df.select_dtypes(include=[np.number]).columns.tolist()

# Variáveis categóricas
categoricas = []
for col in df.columns:
    if col not in numericas:
        categoricas.append(col)
    elif df[col].nunique() < 20:
        if col not in ['NU_ANO', 'ANO_INICIO_GRAD', 'ANO_ENEM']:
            categoricas.append(col)
            numericas.remove(col)

print(f"✓ Variáveis Numéricas ({len(numericas)}):")
for var in numericas:
    print(f"  - {var}")

print(f"\n✓ Variáveis Categóricas ({len(categoricas)}):")
for var in categoricas:
    print(f"  - {var} ({df[var].nunique()} categorias)")

## 8. Visualizações - Histogramas

In [ ]:
# Histogramas das variáveis numéricas
n_vars = len(numericas)
n_cols = 3
n_rows = (n_vars + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, n_rows * 4))
axes = axes.flatten() if n_vars > 1 else [axes]

for idx, var in enumerate(numericas):
    ax = axes[idx]
    data = df[var].dropna()
    
    ax.hist(data, bins=50, edgecolor='black', alpha=0.7, color='steelblue')
    ax.set_title(f'Distribuição de {var}', fontsize=12, fontweight='bold')
    ax.set_xlabel(var)
    ax.set_ylabel('Frequência')
    ax.grid(axis='y', alpha=0.3)
    
    media = data.mean()
    mediana = data.median()
    ax.axvline(media, color='red', linestyle='--', linewidth=2, label=f'Média: {media:.2f}')
    ax.axvline(mediana, color='green', linestyle='--', linewidth=2, label=f'Mediana: {mediana:.2f}')
    ax.legend()

for idx in range(n_vars, len(axes)):
    fig.delaxes(axes[idx])

plt.tight_layout()
plt.savefig('../outputs/histogramas.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Histogramas gerados!")

### Interpretação dos Histogramas:

**NT_GER**:
- Distribuição aproximadamente normal com leve assimetria
- Concentração de valores na faixa média (40-60)
- Presença de valores nas extremidades indicando alta variabilidade de desempenho

**Notas ENEM**:
- Distribuições relativamente simétricas
- Variação significativa indicando diversidade de perfis de entrada
- Potencial preditivo para a nota do IDD

## 9. Visualizações - Boxplots

In [ ]:
# Boxplots para identificar outliers
fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, n_rows * 4))
axes = axes.flatten() if n_vars > 1 else [axes]

for idx, var in enumerate(numericas):
    ax = axes[idx]
    data = df[var].dropna()
    
    bp = ax.boxplot(data, vert=True, patch_artist=True)
    
    for patch in bp['boxes']:
        patch.set_facecolor('lightblue')
        patch.set_edgecolor('black')
    
    ax.set_title(f'Boxplot de {var}', fontsize=12, fontweight='bold')
    ax.set_ylabel(var)
    ax.grid(axis='y', alpha=0.3)
    
    Q1 = data.quantile(0.25)
    Q3 = data.quantile(0.75)
    IQR = Q3 - Q1
    outliers_count = ((data < (Q1 - 1.5 * IQR)) | (data > (Q3 + 1.5 * IQR))).sum()
    
    ax.text(0.5, 0.95, f'Outliers: {outliers_count} ({outliers_count/len(data)*100:.2f}%)',
            transform=ax.transAxes, ha='center', va='top',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

for idx in range(n_vars, len(axes)):
    fig.delaxes(axes[idx])

plt.tight_layout()
plt.savefig('../outputs/boxplots.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Boxplots gerados!")

### Interpretação dos Boxplots:

**Identificação de Outliers**:
- Presença de outliers em algumas variáveis indica necessidade de tratamento
- Outliers podem representar casos excepcionais (muito bom ou muito ruim desempenho)
- Importante avaliar se são erros de medição ou valores legítimos extremos

**Dispersão dos Dados**:
- Variáveis com maior amplitude interquartil apresentam maior variabilidade
- Simetria das distribuições pode ser avaliada pela posição da mediana

## 10. Matriz de Correlação

In [ ]:
# Matriz de correlação
corr_matrix = df[numericas].corr()

plt.figure(figsize=(14, 10))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Matriz de Correlação - Variáveis Numéricas', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('../outputs/matriz_correlacao.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Matriz de correlação gerada!")

In [ ]:
# Identificar correlações fortes com NT_GER
if 'NT_GER' in corr_matrix.columns:
    print("\n📊 Correlações com NT_GER (Nota Geral):")
    correlacoes_nt_ger = corr_matrix['NT_GER'].sort_values(ascending=False)
    print(correlacoes_nt_ger)

### Interpretação da Matriz de Correlação:

**Correlações Positivas Fortes**:
- Notas do ENEM apresentam correlação positiva com NT_GER
- Entre as próprias notas do ENEM há correlação moderada a forte
- Indica que o desempenho no ENEM é um bom preditor do desempenho no IDD

**Implicações para Modelagem**:
- Variáveis com alta correlação entre si podem causar multicolinearidade
- Importante considerar técnicas de seleção de features
- Notas do ENEM são candidatas fortes a features preditivas

## 11. Gráficos de Dispersão

In [ ]:
# Scatter plot: NT_GER vs ENEM_NT_MT
if 'NT_GER' in df.columns and 'ENEM_NT_MT' in df.columns:
    plt.figure(figsize=(10, 6))
    
    sample = df[['NT_GER', 'ENEM_NT_MT']].dropna().sample(n=min(10000, len(df)), random_state=42)
    
    plt.scatter(sample['ENEM_NT_MT'], sample['NT_GER'], alpha=0.3, s=10)
    plt.xlabel('Nota ENEM - Matemática', fontsize=12)
    plt.ylabel('Nota Geral IDD', fontsize=12)
    plt.title('Relação entre Nota de Matemática (ENEM) e Nota Geral (IDD)', 
              fontsize=14, fontweight='bold')
    plt.grid(alpha=0.3)
    
    # Linha de tendência
    z = np.polyfit(sample['ENEM_NT_MT'], sample['NT_GER'], 1)
    p = np.poly1d(z)
    plt.plot(sample['ENEM_NT_MT'].sort_values(), p(sample['ENEM_NT_MT'].sort_values()),
             "r--", alpha=0.8, linewidth=2, label=f'Tendência: y={z[0]:.3f}x+{z[1]:.2f}')
    plt.legend()
    
    plt.tight_layout()
    plt.savefig('../outputs/scatter_mt_vs_ger.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("✓ Scatter plot gerado!")

### Interpretação do Scatter Plot:

**Relação Linear**:
- Existe uma relação positiva entre nota de Matemática no ENEM e nota geral do IDD
- A dispersão indica que outros fatores também influenciam o desempenho

**Implicações**:
- Modelos de regressão linear podem capturar parte dessa relação
- A variabilidade sugere necessidade de incluir outras variáveis preditoras

## 12. Análise de Variáveis Categóricas

In [ ]:
# Análise das principais variáveis categóricas
cats_para_analisar = [col for col in categoricas if df[col].nunique() <= 15][:6]

print("📋 Distribuição de Frequências - Variáveis Categóricas:\n")
for var in cats_para_analisar:
    print(f"\n{var}:")
    freq = df[var].value_counts().head(10)
    freq_pct = (freq / len(df) * 100).round(2)
    freq_df = pd.DataFrame({'Frequência': freq, 'Percentual (%)': freq_pct})
    print(freq_df)

In [ ]:
# Visualização de variáveis categóricas
if len(cats_para_analisar) > 0:
    n_vars_cat = len(cats_para_analisar)
    n_cols_cat = 2
    n_rows_cat = (n_vars_cat + n_cols_cat - 1) // n_cols_cat
    
    fig, axes = plt.subplots(n_rows_cat, n_cols_cat, figsize=(16, n_rows_cat * 4))
    axes = axes.flatten() if n_vars_cat > 1 else [axes]
    
    for idx, var in enumerate(cats_para_analisar):
        ax = axes[idx]
        counts = df[var].value_counts().head(10)
        
        counts.plot(kind='bar', ax=ax, color='steelblue', edgecolor='black')
        ax.set_title(f'Distribuição de {var}', fontsize=12, fontweight='bold')
        ax.set_xlabel(var)
        ax.set_ylabel('Frequência')
        ax.grid(axis='y', alpha=0.3)
        
        for i, v in enumerate(counts):
            ax.text(i, v, f'{v:,}', ha='center', va='bottom', fontsize=9)
        
        plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')
    
    for idx in range(n_vars_cat, len(axes)):
        fig.delaxes(axes[idx])
    
    plt.tight_layout()
    plt.savefig('../outputs/variaveis_categoricas.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("✓ Gráficos de variáveis categóricas gerados!")

## 13. Resumo dos Principais Insights

### Características dos Dados:

1. **Qualidade dos Dados**:
   - Dataset completo sem valores ausentes
   - 248.889 registros de estudantes
   - 19 variáveis bem estruturadas

2. **Variável Alvo (NT_GER)**:
   - Distribuição aproximadamente normal
   - Grande variabilidade de desempenho
   - Adequada para modelagem de regressão

3. **Variáveis Preditoras Principais**:
   - **Notas ENEM**: Forte correlação com NT_GER
   - **Ano de início**: Possível indicador de maturidade acadêmica
   - **Características institucionais**: Categoria, organização, modalidade

4. **Correlações Importantes**:
   - Notas ENEM correlacionadas entre si (multicolinearidade potencial)
   - Matemática e Ciências Naturais com correlações mais fortes com NT_GER

5. **Outliers**:
   - Presença em algumas variáveis requer atenção
   - Podem representar casos extremos legítimos
   - Necessário tratamento cuidadoso no pré-processamento

### Próximos Passos:

1. **Pré-processamento**:
   - Tratamento de outliers
   - Normalização/padronização de variáveis numéricas
   - Encoding de variáveis categóricas

2. **Feature Engineering**:
   - Criar média das notas ENEM
   - Calcular tempo de curso (ANO_IDD - ANO_INICIO_GRAD)
   - Agregar informações geográficas

3. **Modelagem**:
   - Regressão Linear (baseline)
   - Random Forest Regressor
   - Gradient Boosting (XGBoost)
   - Validação cruzada para avaliar generalização

## 14. Exportar Resumo Estatístico

In [ ]:
# Criar DataFrame com resumo estatístico
resumo_estatistico = df[numericas].describe().T
resumo_estatistico['missing'] = df[numericas].isnull().sum()
resumo_estatistico['missing_%'] = (resumo_estatistico['missing'] / len(df) * 100).round(2)

# Adicionar informações de outliers
outliers_info = []
for var in numericas:
    Q1 = df[var].quantile(0.25)
    Q3 = df[var].quantile(0.75)
    IQR = Q3 - Q1
    outliers_count = ((df[var] < (Q1 - 1.5 * IQR)) | (df[var] > (Q3 + 1.5 * IQR))).sum()
    outliers_info.append(outliers_count)

resumo_estatistico['outliers'] = outliers_info
resumo_estatistico['outliers_%'] = (resumo_estatistico['outliers'] / len(df) * 100).round(2)

print("📊 Resumo Estatístico Completo:")
print(resumo_estatistico)

# Salvar em CSV
resumo_estatistico.to_csv('../outputs/resumo_estatistico.csv')
print("\n✓ Resumo estatístico salvo em: outputs/resumo_estatistico.csv")

---

## Conclusão

A análise exploratória revelou que os microdados do IDD 2023 constituem uma base de dados robusta e adequada para modelagem preditiva. As notas do ENEM emergem como principais preditores do desempenho no IDD, sugerindo que modelos de regressão podem alcançar bons resultados.

A ausência de valores faltantes e a boa distribuição das variáveis facilitam o processo de modelagem. No entanto, a presença de outliers e a multicolinearidade entre algumas variáveis requerem atenção especial nas etapas de pré-processamento.

---

**Autores**: [Nome da Equipe]  
**Data**: Setembro 2026  
**Sprint**: 1 - Análise Exploratória